# Student Placement Prediction - Training Notebook

End-to-end pipeline:
1. Load and inspect dataset
2. Preprocess (missing values, encoding, scaling)
3. Train Random Forest classifier
4. Evaluate (accuracy, precision, recall, F1, confusion matrix)
5. Save `model.pkl`, `scaler.pkl`, `encoder.pkl`, `meta.pkl` for the Flask backend

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
warnings.filterwarnings('ignore')

## 1. Load and analyse

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'student_placement_prediction_dataset_2026.csv')
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.dtypes)
print('\nMissing values per column:\n', df.isnull().sum())

In [ ]:
if 'student_id' in df.columns:
    df = df.drop(columns=['student_id'])
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns:', categorical_cols)
for col in categorical_cols:
    print(col, '->', df[col].unique().tolist())

## 2. Preprocess

In [ ]:
target_col = 'placement_status'
drop_cols = [target_col, 'salary_package_lpa']
y_raw = df[target_col].copy()
X = df.drop(columns=drop_cols)

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_feature_cols = [c for c in categorical_cols if c in X.columns]

for col in numeric_cols:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())
for col in cat_feature_cols:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].mode().iloc[0])

feature_encoders = {}
for col in cat_feature_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    feature_encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw.astype(str))
print('Target classes:', list(target_encoder.classes_))

feature_columns = X.columns.tolist()
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
print('Processed feature matrix shape:', X.shape)

## 3. Train Random Forest

Random Forest is robust to feature scaling, captures non-linear interactions, and rarely overfits with sensible hyperparameters - a strong default for tabular classification.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, min_samples_split=5,
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

## 4. Evaluate

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred, average='weighted'), 4))
print('Recall   :', round(recall_score(y_test, y_pred, average='weighted'), 4))
print('F1-score :', round(f1_score(y_test, y_pred, average='weighted'), 4))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\n', classification_report(y_test, y_pred, target_names=target_encoder.classes_))

## 5. Save artifacts for the Flask backend

In [ ]:
BACKEND_DIR = os.path.join('..', 'backend')
os.makedirs(BACKEND_DIR, exist_ok=True)
meta = {
    'feature_columns': feature_columns,
    'numeric_cols': numeric_cols,
    'cat_feature_cols': cat_feature_cols,
    'target_classes': list(target_encoder.classes_),
    'categorical_options': {col: list(map(str, feature_encoders[col].classes_))
                            for col in cat_feature_cols},
}
with open(os.path.join(BACKEND_DIR, 'model.pkl'), 'wb') as f: pickle.dump(model, f)
with open(os.path.join(BACKEND_DIR, 'scaler.pkl'), 'wb') as f: pickle.dump(scaler, f)
with open(os.path.join(BACKEND_DIR, 'encoder.pkl'), 'wb') as f:
    pickle.dump({'features': feature_encoders, 'target': target_encoder}, f)
with open(os.path.join(BACKEND_DIR, 'meta.pkl'), 'wb') as f: pickle.dump(meta, f)
print('Saved model.pkl, scaler.pkl, encoder.pkl, meta.pkl into', BACKEND_DIR)